# EquityLens walkthrough

Value a company from its SEC filings, step by step. Run `python src/run_valuation.py AAPL` once first so the SEC data is cached.

In [ ]:
import sys, os
os.chdir(os.path.abspath('..'))
sys.path.insert(0, 'src')
from edgar_client import latest_years
from dcf import load_assumptions, run_dcf
import comps as comps_mod, simulate as sim

TICKER = 'AAPL'
a = load_assumptions()
company, rows = latest_years(TICKER, n=5)
company

## 1. The raw fundamentals (straight from SEC EDGAR)

In [ ]:
import pandas as pd
df = pd.DataFrame(rows).set_index('year')
df[['revenue','operating_income','d_and_a','capex','net_income']] / 1e9

## 2. The DCF

In [ ]:
d = run_dcf(company, TICKER, rows, a)
print(f"WACC {d['wacc']:.1%}  |  value/share ${d['value_per_share']:,.0f}  |  price ${d['market_price']}")
pd.DataFrame(d['schedule'])[['year','revenue','fcf']].assign(revenue=lambda x: x.revenue/1e9, fcf=lambda x: x.fcf/1e9)

## 3. Range: Monte Carlo + comps + reverse DCF

In [ ]:
mc = sim.monte_carlo(rows, a, d['wacc'], d['terminal_growth'])
cmp = comps_mod.implied_values(TICKER, rows, comps_mod.load_peers())
iw = sim.implied_wacc(rows, a, d['terminal_growth'], d['market_price'])
print(f"Monte Carlo P10-P90: ${mc['p10']:,.0f} - ${mc['p90']:,.0f}")
print(f"Comps low-high:      ${cmp['low']:,.0f} - ${cmp['high']:,.0f}")
print(f"Price implies WACC:  {iw['wacc']:.1%}" if iw['reachable'] else 'price outside DCF band')

## 4. Takeaway

A history-based DCF and comps both land below the market price, so Apple is priced for more than five years of fundamentals imply. The reverse DCF quantifies it: the price implies a ~5.5% discount rate. See `reports/valuation_note_AAPL.md` for the write-up.